# Demo 1: ER modell → Star schema
## Normalizált séma → Dimenziós modell

**Cél:** Felépítünk egy normalizált (3NF) sémát DuckDB-ben, majd átalakítjuk star sémává – megmutatva az ETL pipeline lépéseit.

**Kapcsolódó diák:** 3–12 (Adatmodellezési módszertanok, ER modell, Normalizáció, Kimball)

## 0. Előkészítés – csomagok és kapcsolat

### Miért DuckDB az OLTP demóhoz is?

Valós környezetben az OLTP rendszer PostgreSQL, MySQL vagy Oracle lenne.
A demóban DuckDB-t használunk **mindkét réteghez** (OLTP és OLAP), mert:
- Nincs szükség külön Docker service-re vagy hálózati kapcsolatra
- A DuckDB teljes SQL DDL-t támogat (PRIMARY KEY, FOREIGN KEY, UNIQUE, CHECK)
- A 3NF vs star schema különbség a sémastruktúrában és a lekérdezési mintákban mutatkozik meg, nem a motortól függ

A `duckdb.connect()` egy in-process adatbázist hoz létre – pontosan úgy működik, mint egy SQLite kapcsolat, de analitikai és teljes ANSI SQL képességekkel.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "duckdb", "pandas", "faker"], check=True)

import duckdb
import pandas as pd
from faker import Faker
import random, time
from datetime import datetime, timedelta

fake = Faker('hu_HU')
random.seed(42)
Faker.seed(42)

# DuckDB kapcsolat – memóriában (in-process, nincs szerver szükséges)
con = duckdb.connect()

print(f"DuckDB: {duckdb.__version__}  |  Pandas: {pd.__version__}")
print("Kapcsolat létrehozva: OK")

## 1. rész: Normalizált (3NF) séma – az OLTP alap

### Miért szükséges a normalizáció?

A **Third Normal Form (3NF)** az OLTP rendszerek tervezési alapelve. Három feltételt támaszt:
1. **1NF**: minden mező atomikus értéket tartalmaz (nincs tömb, nincs ismétlő csoport)
2. **2NF**: minden nem-kulcs attribútum teljesen függ az összetett PK-tól (nincs részleges függőség)
3. **3NF**: nincs tranzitív függőség – nem-kulcs attribútum nem függhet másik nem-kulcs attribútumon keresztül a PK-tól

**Példa tranzitív függőségre (3NF megsértése):**
Ha `order_items`-ben tárolnánk a `category_name`-t, az `category_id`-n keresztül függne `product_id`-tól.
Eredmény: ha a kategórianév megváltozik, **minden** order_items sorban frissíteni kell → anomáliák.

**Táblastruktúra:**
```
categories ←── products ←── order_items ──→ orders ──→ customers
```
Minden nyíl egy FOREIGN KEY: az adat egyszer él, hivatkozásokkal kötjük össze.

In [ ]:
# 3NF séma létrehozása DuckDB-ben
# A DuckDB teljes SQL DDL-t támogat, beleértve a FOREIGN KEY kényszereket
schema_sql = """
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS categories;
DROP TABLE IF EXISTS customers;

CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    name        VARCHAR(50)   NOT NULL,
    department  VARCHAR(50)   NOT NULL
);

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        VARCHAR(100)  NOT NULL,
    email       VARCHAR(150)  NOT NULL,
    city        VARCHAR(100),
    segment     VARCHAR(30)
);

CREATE TABLE products (
    product_id  INTEGER PRIMARY KEY,
    name        VARCHAR(100)  NOT NULL,
    category_id INTEGER REFERENCES categories(category_id),
    price       DECIMAL(10,2) NOT NULL
);

CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER REFERENCES customers(customer_id),
    order_date  DATE    NOT NULL,
    status      VARCHAR(20) DEFAULT 'pending'
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id      INTEGER REFERENCES orders(order_id),
    product_id    INTEGER REFERENCES products(product_id),
    quantity      INTEGER       NOT NULL,
    unit_price    DECIMAL(10,2) NOT NULL
);
"""

con.execute(schema_sql)
print("3NF séma létrehozva (5 tábla).")
print(con.execute("SHOW TABLES").df().to_string(index=False))

## 1b. Adatok generálása – Faker és seed

### Reprodukálható szintetikus adatok

A **Faker** könyvtár valósághű, de kitalált adatokat generál. Fontos technikai részletek:

- `Faker('hu_HU')`: lokalizáció – magyar neveket, városokat ad
- `random.seed(42)` és `Faker.seed(42)`: **rögzített seed** → minden futásnál azonos adatok keletkeznek, ami reprodukálhatóvá teszi a demót
- Az egyedi email-cím garantálása: egy `seen_emails` halmazban tartjuk nyilván az eddig generált e-maileket

### Adatmérnöki döntés: batch insert vs. egyenkénti INSERT

Az adatok `executemany()`-vel töltjük be, ami egyetlen hálózati round-trip alatt betölti az összes sort.
Egyenkénti `execute()` 1000 ügyféllel ~1000× lassabb lenne.

In [ ]:
# Kategóriák betöltése
CATEGORIES = [
    (1, "Elektronika", "Technológia"),
    (2, "Laptop",      "Technológia"),
    (3, "Telefon",     "Technológia"),
    (4, "Bútor",       "Otthon"),
    (5, "Irodaszer",   "Irodai"),
    (6, "Könyv",       "Kultúra"),
]

con.executemany("INSERT INTO categories VALUES (?, ?, ?)", CATEGORIES)
print(f"{len(CATEGORIES)} kategória betöltve.")

In [ ]:
# 1000 ügyfél betöltése
SEGMENTS = ["Consumer", "Corporate", "Home Office"]
seen_emails = set()
customers = []
cid = 1
while len(customers) < 1000:
    email = fake.email()
    if email in seen_emails:
        continue
    seen_emails.add(email)
    customers.append((cid, fake.name(), email, fake.city(), random.choice(SEGMENTS)))
    cid += 1

con.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?)", customers)
print(f"{len(customers)} ügyfél betöltve.")

In [ ]:
# 200 termék betöltése
products_data = [
    (i, fake.catch_phrase()[:100], random.randint(1, len(CATEGORIES)),
     round(random.uniform(5, 2000), 2))
    for i in range(1, 201)
]
con.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", products_data)
print(f"{len(products_data)} termék betöltve.")

### Rendelések generálása – N:M kapcsolat kezelése

Az `orders` ↔ `products` N:M kapcsolat: egy rendelésben több termék lehet, egy termék több rendelésben szerepelhet.
Ezt az `order_items` **kapcsolótábla** (bridge/junction table) oldja fel – ez a 3NF standard megoldása.

**Adat méretezés:** 5000 rendelés × átlag 3 tétel = ~15 000 order_items sor.
Ez realisztikus: egy kis webshop napi 100-200 rendelése 3 évre vetítve.

In [ ]:
# 5000 rendelés + tételek generálása (1–5 tétel/rendelés)
base_date = datetime(2022, 1, 1)
orders_data = []
items_data  = []
item_id = 1

for oid in range(1, 5001):
    order_date = base_date + timedelta(days=random.randint(0, 730))
    status = random.choice(["pending", "shipped", "delivered"])
    orders_data.append((oid, random.randint(1, 1000), order_date.date(), status))

    for pid in random.sample(range(1, 201), random.randint(1, 5)):
        qty   = random.randint(1, 10)
        price = products_data[pid - 1][3]   # az ár a products_data listából
        items_data.append((item_id, oid, pid, qty, price))
        item_id += 1

con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders_data)
con.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)", items_data)

print(f"{len(orders_data):,} rendelés betöltve.")
print(f"{len(items_data):,} rendelési tétel betöltve.")

## 2. rész: A normalizált séma ellenőrzése

### Mit keresünk itt?

A 3NF egyik legfontosabb tesztkérdése: **tartalmaz-e a kapcsolótábla leíró adatot?**

Az `order_items` táblában **csak** az alábbiak lehetnek:
- Hivatkozások (`order_id`, `product_id`) – FK-k
- A kapcsolat saját attribútumai (`quantity`, `unit_price`) – a rendelési tételhez tartozó tények

Ha `category_name` vagy `customer_city` is itt lenne, az **tranzitív függőséget** jelent – 3NF sérül.

In [ ]:
# Milyen oszlopok vannak az order_items táblában?
# Helyes 3NF esetén: csak order_id FK, product_id FK és a kapcsolat mérőszámai (qty, price)
cols = con.execute("DESCRIBE order_items").df()
print("order_items oszlopai:")
print(cols[["column_name", "column_type"]].to_string(index=False))
print("\n→ Nincs customer_name, city, category_name – nincsenek tranzitív függőségek.")

print()
for tbl in ["categories", "customers", "products", "orders", "order_items"]:
    cnt = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<15}: {cnt:>6,} sor")

### 3NF analitikai lekérdezés – a JOIN-ok ára

Az alábbi lekérdezés **4 tábla JOIN-ját** igényli, hogy a bevételt kategóriánként aggregálja.
Ez az OLTP séma analitikánál jellemző hátrány:

```
order_items → orders       (rendelési tétel → rendelés dátuma)
order_items → products     (rendelési tétel → termék)
products    → categories   (termék → kategória neve)
```

Az OLAP star schemában ugyanez **3 tábla JOIN**-nal megoldható,
mert a `dim_product` már tartalmazza a `category` mezőt (denormalizáció).

In [ ]:
# Analitikai lekérdezés 3NF sémán: havi bevétel kategóriánként
# 4 JOIN szükséges, mert az adatok szét vannak osztva a normalizált táblák között
query_3nf = """
SELECT
    YEAR(o.order_date)                           AS year,
    MONTH(o.order_date)                          AS month,
    c.name                                       AS category,
    ROUND(SUM(oi.quantity * oi.unit_price), 0)   AS revenue
FROM order_items  oi
JOIN orders       o  ON o.order_id    = oi.order_id
JOIN products     p  ON p.product_id  = oi.product_id
JOIN categories   c  ON c.category_id = p.category_id
GROUP BY 1, 2, 3
ORDER BY 1, 2, revenue DESC
LIMIT 12
"""

t0 = time.time()
result_3nf = con.execute(query_3nf).df()
t1 = time.time()

print(f"3NF lekérdezés (4 JOIN): {(t1-t0)*1000:.1f} ms")
print(result_3nf.to_string(index=False))

## 3. rész: Star schema felépítése DuckDB-ben

### Az ETL pipeline Transform fázisa

A star schema felépítése egy **ETL (Extract–Transform–Load)** pipeline:

1. **Extract**: kiolvasás a 3NF forrásból (már megtörtént – adatok a DuckDB OLTP táblákban)
2. **Transform**: típuskonverziók, surrogate kulcsok generálása, denormalizáció
3. **Load**: céltáblák (dim_*, fact_*) feltöltése

### Kimball-módszer rétegei

| Réteg | Táblák | Szerepük |
|-------|--------|---------|
| Dimenzió | `dim_date`, `dim_customer`, `dim_product` | Szűrési és csoportosítási kontextus |
| Tény | `fact_orders` | Mérőszámok (revenue, quantity) + FK hivatkozások |

### dim_date: miért generált, nem OLTP-ből?

A dátum dimenzió **nem az OLTP forrásból** jön – generate_series-szel hozzuk létre.
Ennek oka: az OLTP forrásban csak a ténylegesen létező rendelési dátumok vannak.
A BI eszközök viszont **minden napot** meg akarnak jeleníteni, még a 0 rendeléssel bírókat is.
Ezen felül a dim_date tartalmaz levezetett attribútumokat: `is_weekend`, `month_name`, `quarter` – ezeket nem célszerű minden ténytáblában ismételni.

In [ ]:
# dim_date: előre generált dátum dimenzió 2022–2023-ra
# Az SK = YYYYMMDD integer formátum – emberi szemmel is olvasható, mégis gyors JOIN
con.execute("""
    CREATE OR REPLACE TABLE dim_date AS
    SELECT
        CAST(strftime(d::DATE, '%Y%m%d') AS INTEGER) AS date_sk,
        d::DATE                                        AS full_date,
        YEAR(d::DATE)                                  AS year,
        QUARTER(d::DATE)                               AS quarter,
        MONTH(d::DATE)                                 AS month,
        MONTHNAME(d::DATE)                             AS month_name,
        DAYOFWEEK(d::DATE)                             AS day_of_week,
        DAYNAME(d::DATE)                               AS day_name,
        DAYOFWEEK(d::DATE) IN (1, 7)                   AS is_weekend
    FROM (
        SELECT DATE '2022-01-01' + INTERVAL (range) DAY AS d
        FROM range(0, 730)   -- 2022–2023, 2 év
    ) dates
""")

cnt = con.execute("SELECT COUNT(*) FROM dim_date").fetchone()[0]
print(f"dim_date: {cnt:,} sor")
print(con.execute("SELECT * FROM dim_date LIMIT 3").df().to_string(index=False))

### Surrogate kulcs (SK) vs. természetes kulcs (NK)

Ez az egyik legfontosabb döntés a DWH tervezésben.

| | Természetes kulcs (NK) | Surrogate kulcs (SK) |
|--|------------------------|----------------------|
| Forrás | Az OLTP forrásrendszerből jön | A DWH generálja (ROW_NUMBER) |
| Stabilitás | Változhat (pl. cég felvásárlás) | Sosem változik |
| JOIN teljesítmény | Lehet string, UUID – lassabb | INTEGER – leggyorsabb |
| SCD2 | Nem kezeli a verziókat | Minden verzióhoz új SK |

**Döntés:** a dim táblák SK-t kapnak a JOIN-okhoz; az NK megmarad visszakereshetőséghez.
`ROW_NUMBER() OVER (ORDER BY customer_id)` generálja az SK-t – folyamatos egész, determinisztikus.

In [ ]:
# dim_customer: ügyféldimenzió surrogate kulccsal
# ROW_NUMBER() generálja az SK-t – független a forrásrendszer ID-jától
# customer_nk = natural key = az eredeti azonosító (megtartjuk visszakereséshez)
con.execute("""
    CREATE OR REPLACE TABLE dim_customer AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_sk,
        customer_id                               AS customer_nk,
        name, email, city, segment
    FROM customers
""")

print(f"dim_customer: {con.execute('SELECT COUNT(*) FROM dim_customer').fetchone()[0]:,} sor")
print(con.execute("SELECT customer_sk, customer_nk, name, city, segment FROM dim_customer LIMIT 5").df().to_string(index=False))

### Denormalizáció döntés: miért kerül category + department a dim_product-ba?

Az OLTP-ben a `products` tábla csak `category_id` FK-t tartalmaz – a kategória neve a `categories` táblában van.
A star schemában ezt **denormalizáljuk**: a `category` és `department` mezők bekerülnek a `dim_product`-ba.

**Miért?**
- BI lekérdezésnél elegendő a `fact_orders JOIN dim_product` – nem kell a categories tábla
- A redundancia (a category neve megismétlődik sok sorban) elfogadott: a DWH-ban az adat általában ritkán változik
- Modern columnar storage-ban a redundancia tárhelyköltsége minimális (oszlopos tömörítés)

**Mikor NEM érdemes denormalizálni?**
Ha a dimenziótábla nagyon nagy és sok leíró attribútuma van → snowflake schema (normalizált dimenzió)

In [ ]:
# dim_product: termékdimenzió
# FONTOS: a category és department is bekerül a dim_product-ba → DENORMALIZÁLÁS
# Ez redundanciát jelent (a category neve ismétlődik), de elkerüli a JOIN-t az analitikánál
con.execute("""
    CREATE OR REPLACE TABLE dim_product AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY p.product_id) AS product_sk,
        p.product_id                               AS product_nk,
        p.name   AS product_name,
        c.name   AS category,
        c.department,
        p.price
    FROM products  p
    JOIN categories c ON p.category_id = c.category_id
""")

print(f"dim_product: {con.execute('SELECT COUNT(*) FROM dim_product').fetchone()[0]:,} sor")
print("\nMegjegyzés: category és department egyazon dim_product sorban = denormalizáció")
print(con.execute("SELECT product_sk, product_name, category, department FROM dim_product LIMIT 5").df().to_string(index=False))

## 4. rész: Fact tábla feltöltése

### A fact tábla szerkezeti döntései

**Grain (szemcsézettség) deklaráció:** `fact_orders` grainje = **1 sor = 1 order_items sor** (1 rendelési tétel).
Ez a legfontosabb döntés a Kimball-módszerben – a grain meghatározza, mi kérdezhető le közvetlenül.

**NK → SK csere:** az `order_items.customer_id` természetes kulcsot `customer_sk`-ra kell cserélni.
Ez a JOIN a DIM táblára: `JOIN dim_customer dc ON dc.customer_nk = o.customer_id`.

**Miért tároljuk az `unit_price`-t a fact táblában?**
Mert az ár **az eladás pillanatában érvényes érték** – a `dim_product.price` az aktuális árat tárolja,
amely megváltozhat. A ténytáblában az historikus ár rögzített mérőszám.

In [ ]:
# fact_orders feltöltése – Transform + Load fázis
# A JOIN-ok cserélik a természetes kulcsokat (customer_id, product_id) surrogate kulcsokra (customer_sk, product_sk)
con.execute("""
    CREATE OR REPLACE TABLE fact_orders AS
    SELECT
        ROW_NUMBER() OVER ()                                             AS order_line_sk,
        CAST(strftime(o.order_date::DATE, '%Y%m%d') AS INTEGER)         AS date_sk,
        dc.customer_sk,
        dp.product_sk,
        oi.quantity,
        oi.unit_price,
        ROUND(oi.quantity * oi.unit_price, 2)                            AS revenue
    FROM order_items  oi
    JOIN orders       o   ON o.order_id     = oi.order_id
    JOIN dim_customer dc  ON dc.customer_nk  = o.customer_id
    JOIN dim_product  dp  ON dp.product_nk   = oi.product_id
    JOIN dim_date     dd  ON dd.date_sk = CAST(strftime(o.order_date::DATE, '%Y%m%d') AS INTEGER)
""")

cnt = con.execute("SELECT COUNT(*) FROM fact_orders").fetchone()[0]
print(f"fact_orders: {cnt:,} sor")
print(con.execute("SELECT * FROM fact_orders LIMIT 5").df().to_string(index=False))

## 5. rész: Analitika összehasonlítása

### Mit mutat ez az összehasonlítás?

Ugyanaz az üzleti kérdés (*havi bevétel kategóriánként*) két sémamodellen:

| Séma | JOIN-ok | Miért? |
|------|---------|--------|
| 3NF | 4 tábla | order_items → orders → products → categories |
| Star | 3 tábla | fact_orders → dim_date + dim_product (category már benne) |

A DuckDB esetén a sebesség különbség kis adatokon nem szignifikáns.
**Produkciós méretű adatnál** (százmillió sor): a star schema lekérdezés 2–10× gyorsabb lehet,
mert kevesebb JOIN és a columnar engine az aggregálást SIMD vektorizálással végzi.

In [ ]:
# Star schema lekérdezés – 3 tábla JOIN (dim_product már tartalmazza a category-t)
star_query = """
SELECT
    d.year,
    d.month_name,
    p.category,
    c.segment,
    ROUND(SUM(f.revenue), 0)  AS total_revenue,
    SUM(f.quantity)           AS total_qty,
    COUNT(*)                  AS order_lines
FROM fact_orders  f
JOIN dim_date     d ON d.date_sk     = f.date_sk
JOIN dim_customer c ON c.customer_sk = f.customer_sk
JOIN dim_product  p ON p.product_sk  = f.product_sk
GROUP BY d.year, d.month_name, p.category, c.segment
ORDER BY d.year, total_revenue DESC
LIMIT 10
"""

t0 = time.time()
result_star = con.execute(star_query).df()
t1 = time.time()

print(f"Star schema lekérdezés: {(t1-t0)*1000:.1f} ms")
print(result_star.to_string(index=False))

In [ ]:
# Összesítő – séma összehasonlítás
print("=" * 62)
print("ÖSSZESÍTŐ: 3NF vs Star schema")
print("=" * 62)
print(f"{'Szempont':<30} {'3NF (OLTP)':<20} {'Star schema (OLAP)'}")
print("-" * 62)
rows = [
    ("Táblák száma",         "5 (normalizált)",    "4 (fact + 3 dim)"),
    ("JOIN-ok analitikánál", "4",                  "3"),
    ("Redundancia",          "Minimális",          "Szándékos (gyors)"),
    ("INSERT / UPDATE",      "Gyors",              "Ritkán módosul"),
    ("Analitika",            "Lassabb JOINok",     "Gyors columnar engine"),
    ("Célja",                "OLTP (operatív)",    "OLAP (analitika)"),
]
for r in rows:
    print(f"{r[0]:<30} {r[1]:<20} {r[2]}")
print("\n→ A star schema nem 'jobb' – más célra optimalizált.")